#### Loader Class

In [ ]:
class Loader:
    def __init__(self):
        pass

    def sink(self):
        pass

### Extractor Class

In [ ]:
class Extractor:
    def __init__(self):
        pass
    

### Transform Class

In [ ]:
class Transform:
    def __init__(self):
        pass
    def transform(self, inputDf):
        pass

#### Workflow

In [ ]:
class Workflow:
    def __init__(self):
        pass

In [ ]:
#pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 395.5 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
import data_generator as dg

### Faker

In [5]:
if __name__ == "__main__":
    dg.generate_data()

2025-04-29 15:10:29 - ecommerce_generator - INFO - Starting e-commerce data generation: 50 events
2025-04-29 15:10:29 - ecommerce_generator - INFO - Generating 50 e-commerce events...
2025-04-29 15:10:29 - ecommerce_generator - INFO - Saved 50 events to ecommerce_data/all_events.csv
2025-04-29 15:10:29 - ecommerce_generator - INFO - Saved batch 1 with 10 events to ecommerce_data/events_batch_1.csv
2025-04-29 15:10:29 - ecommerce_generator - INFO - Saved batch 2 with 10 events to ecommerce_data/events_batch_2.csv
2025-04-29 15:10:29 - ecommerce_generator - INFO - Saved batch 3 with 10 events to ecommerce_data/events_batch_3.csv
2025-04-29 15:10:29 - ecommerce_generator - INFO - Saved batch 4 with 10 events to ecommerce_data/events_batch_4.csv
2025-04-29 15:10:29 - ecommerce_generator - INFO - Saved batch 5 with 10 events to ecommerce_data/events_batch_5.csv
2025-04-29 15:10:29 - ecommerce_generator - INFO - Data generation complete!
2025-04-29 15:10:29 - ecommerce_generator - INFO - Fil

In [ ]:
spark = SparkSession.builder \
    .appName("CSV-PostgreSQL-Pipeline") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.5.1") \
    .getOrCreate()

In [ ]:
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("field1", StringType(), True),
    StructField("field2", DoubleType(), True),
    # Add fields as needed
])

#### Datasource layer

In [ ]:
csv_df = spark.readStream \
    .format("csv") \
    .schema(schema) \
    .option("header", True) \
    .option("path", "/path/to/csv/dir") \
    .option("maxFilesPerTrigger", 1) \
    .load()

In [ ]:
class DataSourceFactory:
    @staticmethod
    def get_source(source_type, spark, config):
        if source_type == "csv":
            return CSVSource(spark, config)
        elif source_type == "json":
            return JSONSource(spark, config)
        # Add more source types as needed

class DataSource(abc.ABC):
    def __init__(self, spark, config):
        self.spark = spark
        self.config = config
    
    @abc.abstractmethod
    def read_stream(self):
        pass

class CSVSource(DataSource):
    def read_stream(self):
        return (self.spark.readStream
                .format("csv")
                .schema(self.config["schema"])
                .option("header", self.config.get("header", True))
                .option("path", self.config["path"])
                .option("maxFilesPerTrigger", self.config.get("maxFilesPerTrigger", 1))
                .load())

#### Transformation

In [ ]:
def clean_data(df):
    return df.na.drop(subset=["id"])

def transform_data(df):
    return df.withColumn("processed_time", current_timestamp())

processed_df = transform_data(clean_data(csv_df))

In [ ]:
class Transformations:
    @staticmethod
    def clean_data(df):
        """Remove nulls and duplicates"""
        return df.na.drop().dropDuplicates(["id"])
    
    @staticmethod
    def enrich_with_metadata(df):
        """Add processing metadata"""
        return (df.withColumn("processing_timestamp", current_timestamp())
                .withColumn("processing_date", current_date())
                .withColumn("batch_id", lit(uuid.uuid4().hex)))
    
    @staticmethod
    def categorize_values(df, column, ranges, category_column):
        """Categorize numeric values into ranges"""
        conditions = []
        values = []
        
        for i, (lower, upper, category) in enumerate(ranges):
            if i == 0:
                conditions.append(col(column) < upper)
            elif i == len(ranges) - 1:
                conditions.append(col(column) >= lower)
            else:
                conditions.append((col(column) >= lower) & (col(column) < upper))
            values.append(category)
        
        return df.withColumn(category_column, when(conditions[0], values[0])
                             .when(conditions[1], values[1])
                             .otherwise(values[2]))
    
    @staticmethod
    def derive_time_dimensions(df, timestamp_col):
        """Extract time dimensions"""
        return (df.withColumn("year", year(col(timestamp_col)))
                .withColumn("month", month(col(timestamp_col)))
                .withColumn("day", dayofmonth(col(timestamp_col)))
                .withColumn("hour", hour(col(timestamp_col)))
                .withColumn("day_of_week", dayofweek(col(timestamp_col))))

#### Datasink 

In [ ]:
def write_to_postgres(batch_df, batch_id):
    batch_df.write \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://localhost:5432/yourdb") \
        .option("dbtable", "your_table") \
        .option("user", "postgres") \
        .option("password", "password") \
        .mode("append") \
        .save()

query = processed_df.writeStream \
    .foreachBatch(write_to_postgres) \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/checkpoints") \
    .start()

In [ ]:
class DataSinkFactory:
    @staticmethod
    def get_sink(sink_type, config):
        if sink_type == "postgres":
            return PostgresSink(config)
        elif sink_type == "console":
            return ConsoleSink(config)
        # Add more sink types as needed

class DataSink(abc.ABC):
    def __init__(self, config):
        self.config = config
    
    @abc.abstractmethod
    def write_stream(self, df):
        pass

class PostgresSink(DataSink):
    def write_stream(self, df):
        return (df.writeStream
                .foreachBatch(self._write_to_postgres)
                .outputMode(self.config.get("output_mode", "append"))
                .option("checkpointLocation", self.config["checkpoint_dir"])
                .trigger(processingTime=self.config.get("trigger_interval", "10 seconds"))
                .start())
    
    def _write_to_postgres(self, batch_df, batch_id):
        try:
            # Create a connection pool for better performance
            connection_properties = {
                "url": self.config["jdbc_url"],
                "user": self.config["user"],
                "password": self.config["password"],
                "driver": "org.postgresql.Driver"
            }
            
            # Write to PostgreSQL with optimized settings
            (batch_df.write
                .jdbc(
                    url=connection_properties["url"],
                    table=self.config["table"],
                    mode="append",
                    properties=connection_properties
                ))
            
            # Log success with metrics
            print(f"Batch {batch_id}: Successfully wrote {batch_df.count()} records")
            
        except Exception as e:
            # Enhanced error handling
            print(f"Error writing batch {batch_id}: {str(e)}")
            
            # Save failed batch for later retry
            if self.config.get("save_failed_batches", False):
                failed_path = f"{self.config.get('failed_batches_dir', '/tmp/failed')}/batch_{batch_id}"
                batch_df.write.parquet(failed_path)
                print(f"Saved failed batch to {failed_path}")

#### Pipeline Orchestration

In [ ]:
query.awaitTermination()

In [ ]:
class StreamingPipeline:
    def __init__(self, app_name, source_type, source_config, sink_type, sink_config):
        self.app_name = app_name
        self.source_type = source_type
        self.source_config = source_config
        self.sink_type = sink_type
        self.sink_config = sink_config
        self.spark = self._create_spark_session()
        self.source = None
        self.sink = None
        self.query = None
        self.transformation_steps = []
        
        # Configure logging
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger(self.app_name)
    
    def _create_spark_session(self):
        # Enhanced Spark configuration for better performance
        return (SparkSession.builder
                .appName(self.app_name)
                .config("spark.jars.packages", "org.postgresql:postgresql:42.5.1")
                .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoints")
                .config("spark.sql.shuffle.partitions", 4)
                .config("spark.sql.streaming.stateStore.maintenanceInterval", "300s")
                .config("spark.memory.offHeap.enabled", "true")
                .config("spark.memory.offHeap.size", "1g")
                .getOrCreate())
    
    def add_transformation(self, transform_fn):
        """Add a transformation function to the pipeline"""
        self.transformation_steps.append(transform_fn)
        return self
    
    def initialize(self):
        """Initialize pipeline components"""
        self.logger.info("Initializing pipeline")
        self.source = DataSourceFactory.get_source(self.source_type, self.spark, self.source_config)
        self.sink = DataSinkFactory.get_sink(self.sink_type, self.sink_config)
        return self
    
    def run(self):
        """Run the pipeline with monitoring"""
        self.logger.info("Starting pipeline execution")
        start_time = time.time()
        
        try:
            # Read from source
            stream_df = self.source.read_stream()
            processed_df = stream_df
            
            # Apply all transformation steps
            for i, transform_fn in enumerate(self.transformation_steps):
                step_start = time.time()
                processed_df = transform_fn(processed_df)
                self.logger.info(f"Transformation step {i+1} completed in {time.time() - step_start:.2f}s")
            
            # Write to sink
            self.query = self.sink.write_stream(processed_df)
            
            self.logger.info(f"Pipeline started successfully in {time.time() - start_time:.2f}s")
            
            # Add stream monitoring metrics
            if self.config.get("enable_metrics", False):
                self._start_metrics_reporter()
            
            return self.query
        
        except Exception as e:
            self.logger.error(f"Error running pipeline: {str(e)}")
            raise
    
    def _start_metrics_reporter(self):
        """Report metrics about the stream processing"""
        def report_metrics():
            while self.query.isActive:
                metrics = self.query.lastProgress
                if metrics:
                    self.logger.info(f"Input rate: {metrics.get('inputRate', 0):.2f} records/sec")
                    self.logger.info(f"Processing rate: {metrics.get('processingRate', 0):.2f} records/sec")
                    self.logger.info(f"Batch duration: {metrics.get('batchDuration', 0):.2f} ms")
                time.sleep(30)
        
        threading.Thread(target=report_metrics, daemon=True).start()
    
    def stop(self):
        """Gracefully stop the pipeline"""
        if self.query:
            self.logger.info("Stopping pipeline")
            self.query.stop()
            self.logger.info("Pipeline stopped")
        
        if self.spark:
            self.logger.info("Stopping Spark session")
            self.spark.stop()
            self.logger.info("Spark session stopped")

#### Dataquality Monitoring

In [ ]:
class DataQualityChecker:
    @staticmethod
    def check_schema_compliance(df, expected_schema):
        """Verify if dataframe complies with expected schema"""
        actual_fields = set(df.schema.fieldNames())
        expected_fields = set(expected_schema.fieldNames())
        
        missing_fields = expected_fields - actual_fields
        extra_fields = actual_fields - expected_fields
        
        if missing_fields or extra_fields:
            print(f"Schema mismatch! Missing: {missing_fields}, Extra: {extra_fields}")
            return False
        return True
    
    @staticmethod
    def check_data_quality(df):
        """Check for data quality issues"""
        quality_metrics = {
            "row_count": df.count(),
            "null_counts": {field: df.filter(col(field).isNull()).count() 
                           for field in df.schema.fieldNames()},
            "duplicate_count": df.count() - df.dropDuplicates().count()
        }
        return quality_metrics

#### Example Implementation

In [ ]:
# Define schema
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("timestamp", TimestampType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True)
])

# Source config
source_config = {
    "path": "/data/csv_files",
    "schema": schema,
    "header": True,
    "maxFilesPerTrigger": 2
}

# Sink config
sink_config = {
    "jdbc_url": "jdbc:postgresql://localhost:5432/mydb",
    "table": "sales_data",
    "user": "postgres",
    "password": "postgres",
    "checkpoint_dir": "/tmp/checkpoints/sales",
    "save_failed_batches": True,
    "failed_batches_dir": "/tmp/failed_batches"
}

# Custom transformation functions
def calculate_total(df):
    return df.withColumn("total_amount", col("quantity") * col("price"))

def add_time_dimensions(df):
    return Transformations.derive_time_dimensions(df, "timestamp")

# Create and run pipeline with transformations
pipeline = (StreamingPipeline("EnhancedSalesPipeline", "csv", source_config, "postgres", sink_config)
           .initialize()
           .add_transformation(Transformations.clean_data)
           .add_transformation(calculate_total)
           .add_transformation(add_time_dimensions)
           .add_transformation(Transformations.enrich_with_metadata)
           .run())

# Wait for termination
try:
    pipeline.query.awaitTermination()
except KeyboardInterrupt:
    pipeline.stop()